# Визуализация данных

© Гошин Е.В., доцент кафедры технической кибернетики, к.т.н., Самарский университет  
© Петров М.В., старший преподаватель кафедры киберфотоники, Самарский университет

# Лекция 4. Библиотека `Altair`. Продвинутые возможности

## Содержание

1. [Интерактивность и выборки](#41-интерактивность-и-выборки)
2. [Преобразования](#42-преобразования)
3. [Привязки и виджеты](#43-привязки-и-виджеты)
4. [Настройка графиков](#44-настройка-графиков)
5. [Экспорт графиков](#45-экспорт-графиков)

## 4.1 Интерактивность и выборки

Интерактивность `Altair` и его *«грамматика выборок»* (*grammar of selections*) &ndash; одна из уникальных возможностей среди доступных библиотек визуализации.
Последовательно рассмотрим доступные типы выборок и попробуем создание интерактивных графиков.

Существует три базовых типа выборок:

- Интервальные выборки: `alt.selection_interval()`
- Одиночные выборки: `alt.selection_point()`
- Множественные выборки: `alt.selection_point()` + `Shift`

Мы также разберём четыре базовые операции, которые можно выполнять с выборками:
- условные кодировки
- масштабы
- фильтры
- домены

Источники:
 - [Vega-Altair &rarr; Selections: Capturing Chart Interactions](https://altair-viz.github.io/user_guide/interactions/parameters.html#selections-capturing-chart-interactions)
 - [Vega-Altair &rarr; API &rarr; selection_interval](https://altair-viz.github.io/user_guide/generated/api/altair.selection_interval.html)
 - [Vega-Altair &rarr; API &rarr; selection_point](https://altair-viz.github.io/user_guide/generated/api/altair.selection_point.html)
 - [Vega-Altair &rarr; Conditions](https://altair-viz.github.io/user_guide/interactions/parameters.html#conditions)
 - [Vega-Altair &rarr; API &rarr; condition](https://altair-viz.github.io/user_guide/generated/api/altair.condition.html)
 - [Vega-Altair &rarr; Filters](https://altair-viz.github.io/user_guide/interactions/parameters.html)

In [1]:
import altair as alt
from vega_datasets import data

### Базовые взаимодействия: панорамирование, масштабирование, всплывающие подсказки

Мы можем включить их в график без явного использования интерфейса выборок, задействовав метод `interactive()` и кодировку `tooltip`.

Например, для стандартного набора данных об автомобилях выполним следующее:

In [2]:
cars = data.cars()
cars.head()

,Name,Miles_per_Gallon,Cylinders,Displacement,Horsepower,Weight_in_lbs,Acceleration,Year,Origin
0,chevrolet chevelle malibu,18.0,8,307.0,130.0,3504,12.0,1970-01-01,USA
1,buick skylark 320,15.0,8,350.0,165.0,3693,11.5,1970-01-01,USA
2,plymouth satellite,18.0,8,318.0,150.0,3436,11.0,1970-01-01,USA
3,amc rebel sst,16.0,8,304.0,150.0,3433,12.0,1970-01-01,USA
4,ford torino,17.0,8,302.0,140.0,3449,10.5,1970-01-01,USA


In [3]:
alt.Chart(cars).mark_point().encode(
    x='Horsepower:Q',
    y='Miles_per_Gallon:Q',
    color='Origin',
    tooltip= ['Name', 'Cylinders']
).interactive()

alt.Chart(...)

На данном этапе при наведении курсора на точку будет появляться всплывающая подсказка с названием модели автомобиля и количеством цилиндров в двигателе, а щелчок/перетаскивание/прокрутка будут панорамировать и масштабировать область графика.

### Более сложные взаимодействия: выборки

#### Базовый пример выборки: интервал

В качестве примера добавим к графику интервальную выборку.

Начнём с диаграммы рассеяния:

In [4]:
alt.Chart(cars).mark_point().encode(
    x='Horsepower:Q',
    y='Miles_per_Gallon:Q',
    color='Origin'
)

alt.Chart(...)

Чтобы добавить к графику поведение выборки, создадим объект выборки и воспользуемся методом `add_params`:

In [5]:
interval = alt.selection_interval()

alt.Chart(cars).mark_point().encode(
    x='Horsepower:Q',
    y='Miles_per_Gallon:Q',
    color='Origin'
).add_params(
    interval
)

alt.Chart(...)

Это добавляет к графику взаимодействие, позволяющее выделять точки; возможно, самое распространённое применение выборки &ndash; подсветка точек за счёт условной привязки их цвета к результату выборки.

Сделаем это с помощью `alt.condition`:

In [6]:
interval = alt.selection_interval()

alt.Chart(cars).mark_point().encode(
    x='Horsepower:Q',
    y='Miles_per_Gallon:Q',
    color=alt.condition(interval, 'Origin', alt.value('lightgray'))
).add_params(
    interval
)

alt.Chart(...)

Функция `alt.condition` принимает три аргумента: объект выборки, значение, применяемое к точкам внутри выборки, и значение, применяемое к точкам вне выборки. Здесь мы используем `alt.value('lightgray')` чтобы в явном виде указать, что это именно цвет, а не имя столбца данных.

##### Настройка интервальной выборки

Функция `alt.selection_interval()` принимает ряд дополнительных аргументов; например, определяя `encodings`, мы можем контролировать, распространяется ли выборка по оси x, по оси y или по обеим:

In [7]:
interval = alt.selection_interval(encodings=['x'])

alt.Chart(cars).mark_point().encode(
    x='Horsepower:Q',
    y='Miles_per_Gallon:Q',
    color=alt.condition(interval, 'Origin', alt.value('lightgray'))
).add_params(
    interval
)

alt.Chart(...)

In [8]:
interval = alt.selection_interval(encodings=['y'])

alt.Chart(cars).mark_point().encode(
    x='Horsepower:Q',
    y='Miles_per_Gallon:Q',
    color=alt.condition(interval, 'Origin', alt.value('lightgray'))
).add_params(
    interval
)

alt.Chart(...)

Аргумент `empty` позволяет задать, содержит ли пустая выборка все значения или ни одного; при `empty=False` точки по умолчанию отображаются серым:

In [9]:
interval = alt.selection_interval(empty=False)

alt.Chart(cars).mark_point().encode(
    x='Horsepower:Q',
    y='Miles_per_Gallon:Q',
    color=alt.condition(interval, 'Origin', alt.value('lightgray'))
).add_params(
    interval
)

alt.Chart(...)

#### Одиночная выборка

Функция `alt.selection_point()` позволяет пользователю щёлкать по отдельным объектам графика, выбирая их по одному.
Сделаем точки крупнее, чтобы по ним было проще попадать:

In [10]:
point = alt.selection_point()

alt.Chart(cars).mark_circle(size=100).encode(
    x='Horsepower:Q',
    y='Miles_per_Gallon:Q',
    color=alt.condition(point, 'Origin', alt.value('lightgray')),
    tooltip = 'Name'
).add_params(
    point
)

alt.Chart(...)

Одиночная выборка поддерживает и другое поведение; например, мы можем задать `nearest=True` и `on='mouseover'`, чтобы подсветка переходила к ближайшей точке по мере движения курсора:

In [11]:
point = alt.selection_point(on='mouseover', nearest=True)

alt.Chart(cars).mark_circle(size=100).encode(
    x='Horsepower:Q',
    y='Miles_per_Gallon:Q',
    color=alt.condition(point, 'Origin', alt.value('lightgray')),
    tooltip= ['Name', 'Cylinders']
).add_params(
    point
)

alt.Chart(...)

> Проблема `Name undefined` в данном случае возникает из-за бага/особенности самого движка визуализации `Vega-Lite`, когда `nearest=True` используется совместно с `tooltip` на одном и том же слое `mark_circle` (или `mark_point`).  
> Когда добавляется `nearest=True`, `Altair` за кулисами создает невидимый слой сетки (*Voronoi mesh*) для определения ближайшей точки. Из-за этого стандартный `tooltip` теряет связь с конкретной строкой данных (datum) самого графика и пытается прочитать свойства сгенерированной сетки &ndash; из-за чего и выводится `undefined`.  
> Чтобы сохранить и подсветку ближайшей точки (`nearest=True`), и работающие подсказки, нужно разделить график на два слоя:
> 1. Нижний слой &ndash; отвечает за логику `nearest` и изменение цвета (серый/цветной). Сами точки делаются чуть крупнее или прозрачнее, чтобы перехватывать курсор.
> 2. Верхний слой &ndash; выводит стандартные всплывающие подсказки (`tooltip`).

In [12]:
# 1. Создаем точку выбора
point = alt.selection_point(on='mouseover', nearest=True, empty=False)

# 2. Базовый график с логикой выбора и изменения цвета
base = alt.Chart(cars).mark_circle(size=100).encode(
    x='Horsepower:Q',
    y='Miles_per_Gallon:Q',
    color=alt.condition(point, 'Origin:N', alt.value('lightgray'))
).add_params(
    point
)

# 3. Слой исключительно для tooltip (без nearest, чтобы не ломать данные)
tooltips = alt.Chart(cars).mark_circle(size=100, opacity=0).encode(
    x='Horsepower:Q',
    y='Miles_per_Gallon:Q',
    tooltip=['Name:N', 'Cylinders:O'] # Явно указываем типы :N и :O
)

# 4. Накладываем слои друг на друга
chart = alt.layer(base, tooltips)
chart.display()

alt.LayerChart(...)

#### Множественная выборка

Функция `alt.selection_point()` позволяет выделять несколько точек одновременно при удерживании клавиши `Shift`:

In [13]:
point = alt.selection_point()

alt.Chart(cars).mark_circle(size=100).encode(
    x='Horsepower:Q',
    y='Miles_per_Gallon:Q',
    color=alt.condition(point, 'Origin', alt.value('lightgray'))
).add_params(
    point
)

alt.Chart(...)

Параметры `on` и `nearest` также работают для множественных выборок:

In [14]:
point = alt.selection_point(on='mouseover', nearest=True, empty=False)

alt.Chart(cars).mark_circle(size=100).encode(
    x='Horsepower:Q',
    y='Miles_per_Gallon:Q',
    color=alt.condition(point, 'Origin', alt.value('lightgray'))
).add_params(
    point
)

alt.Chart(...)

### Привязка выборок

Выше мы увидели, как `alt.condition` позволяет привязывать выборку к различным аспектам графика. Рассмотрим ещё несколько способов использования выборок.

#### Привязка масштабов

Для интервальной выборки мы также можем привязать выделенную область к масштабам графика:

In [15]:
bind = alt.selection_interval(bind='scales')

alt.Chart(cars).mark_circle(size=100).encode(
    x='Horsepower:Q',
    y='Miles_per_Gallon:Q',
    color='Origin:N'
).add_params(
    bind
)

alt.Chart(...)

По сути, именно это делает метод `chart.interactive()`.

#### Привязка масштабов к другим доменам

Также возможно привязывать масштабы к внешним доменам:

In [16]:
weather = data.seattle_weather()
weather.head()

,date,precipitation,temp_max,temp_min,wind,weather
0,2012-01-01,0.0,12.8,5.0,4.7,drizzle
1,2012-01-02,10.9,10.6,2.8,4.5,rain
2,2012-01-03,0.8,11.7,7.2,2.3,rain
3,2012-01-04,20.3,12.2,5.6,4.7,rain
4,2012-01-05,1.3,8.9,2.8,6.1,rain


In [17]:
base = alt.Chart(weather).mark_rule().encode(
    x='date:T',
    y='temp_min:Q',
    y2='temp_max:Q',
    color='weather:N'
)

base

alt.Chart(...)

In [18]:
chart = base.properties(
    width=800,
    height=300
)

view = chart.properties(
    width=800,
    height=50
)

chart & view

alt.VConcatChart(...)

Добавим интервальную выборку к нижнему графику, которая будет управлять доменом верхнего графика:

In [19]:
interval = alt.selection_interval(encodings=['x'])

base = alt.Chart(weather).mark_rule(size=2).encode(
    x='date:T',
    y='temp_min:Q',
    y2='temp_max:Q',
    color='weather:N'
)

chart = base.encode(
    x=alt.X('date:T', scale=alt.Scale(domain=interval.to_dict()))
).properties(
    width=800,
    height=300
)

view = base.add_params(
    interval
).properties(
    width=800,
    height=50,
)

chart & view

alt.VConcatChart(...)

#### Фильтрация по выборке

В многопанельных визуализациях мы можем использовать результат выборки, чтобы фильтровать другие представления данных.  
Например, ниже приведены диаграмма рассеяния и гистограмма.

In [20]:
scatter = alt.Chart(cars).mark_point().encode(
    x='Horsepower:Q',
    y='Miles_per_Gallon:Q',
    color='Origin:N'
)

hist = alt.Chart(cars).mark_bar().encode(
    x='count()',
    y='Origin',
    color='Origin'
)

scatter & hist

alt.VConcatChart(...)

Добавим фильтрацию данных с помощью `transform_filter`:

In [21]:
interval = alt.selection_interval()

scatter = alt.Chart(cars).mark_point().encode(
    x='Horsepower:Q',
    y='Miles_per_Gallon:Q',
    color=alt.condition(interval, 'Origin:N', alt.value('lightgray'))
).add_params(
    interval
)

hist = alt.Chart(cars).mark_bar().encode(
    x='count()',
    y='Origin',
    color='Origin'
).transform_filter(
    interval
)

scatter & hist

alt.VConcatChart(...)

Аналогично, можно воспользоваться множественной выборкой, чтобы это работало и в обратную сторону (щелчок по столбцу гистограммы будет фильтровать содержимое диаграммы рассеяния).
Добавим это к предыдущему примеру:

In [22]:
click = alt.selection_point(encodings=['color'])

scatter = alt.Chart(cars).mark_point().encode(
    x='Horsepower:Q',
    y='Miles_per_Gallon:Q',
    color='Origin:N'
).transform_filter(
    click
)

hist = alt.Chart(cars).mark_bar().encode(
    x='count()',
    y='Origin',
    color=alt.condition(click, 'Origin', alt.value('lightgray'))
).add_params(
    click
)

scatter & hist

alt.VConcatChart(...)

> Множественный «клик» также доступен с использованием `Shift`.

Рассмотрим ещё один вариант:

In [23]:
selected = alt.selection_point(encodings=['x'])

heatmap = alt.Chart(cars).mark_rect().encode(
    alt.X('Horsepower:Q', bin=True),
    alt.Y('Miles_per_Gallon:Q', bin=True),
    alt.Color('count()',
        scale=alt.Scale(scheme='greenblue'),
        legend=alt.Legend(title='Total Cars')
    )
).properties(
    width=350
)

# Place the circles
circles = heatmap.mark_point().encode(
    alt.ColorValue('grey'),
    alt.Size('count()')
).transform_filter(
    selected
)

# Place the bars
bars = alt.Chart(cars).mark_bar().encode(
    x='Origin:N',
    y='count()',
    color=alt.condition(selected, 'Origin:N', alt.ColorValue("grey"))
).properties(
    width=350
).add_params(selected)

heatmap + circles | bars

alt.HConcatChart(...)

### Итоги по выборкам

- Типы выборок:
  + интервальные выборки: `alt.selection_interval()`
  + одиночные выборки: `alt.selection_point()`
  + множественные выборки: `alt.selection_point()` + `Shift`
  
- Привязки
  + привязка масштабов: перетаскиваем и прокручиваем для взаимодействия с графиком
  + привязка масштабов на другом графике
  + условные кодировки (например, цвет, размер)
  + фильтрация данных

## 4.2 Преобразования

Одним из ключевых этапов в процессе визуализации является преобразование данных.
В `Altair` существует два основных подхода к выполнению трансформаций:

1. Предварительное преобразование данных на стороне Python
2. Преобразования непосредственно в `Altair`/`Vega-Lite`

Источники:
 - [Vega-Altair &rarr; User Guide &rarr; Calculate](https://altair-viz.github.io/user_guide/transform/calculate.html)
 - [Vega-Altair &rarr; User Guide &rarr; Data Transformations](https://altair-viz.github.io/user_guide/transform/index.html)
 - [Vega Expression](https://vega.github.io/vega/docs/expressions/)

### Вычисления

Рассмотрим пример преобразования входных данных, которые представлены в неявной форме.
Набор данных ``population`` содержит агрегированные данные  населения по годам, полу и возрасту, однако пол закодирован числами «1» и «2». Это делает подписи на графиках неинформативными:

In [24]:
population = data.population()
population.head()

,year,age,sex,people
0,1850,0,1,1483789
1,1850,0,2,1450376
2,1850,5,1,1411067
3,1850,5,2,1359668
4,1850,10,1,1260099


In [25]:
alt.Chart(population).mark_bar().encode(
    x='year:O',
    y='sum(people):Q',
    color='sex:N'
)

alt.Chart(...)

Одним из способов решения этой проблемы в `Python` является перекодировка значений с помощью инструментов `pandas`. Например:

In [26]:
population['Пол'] = population['sex'].map({1: 'Мужчины', 2: 'Женщины'})

alt.Chart(population).mark_bar().encode(
    x='year:O',
    y='sum(people):Q',
    color='Пол:N'
)

alt.Chart(...)

Однако, `Altair` часто используется с данными, получаемыми по URL-адресу, и в таком случае предварительная обработка недоступна. В подобных ситуациях целесообразно включить преобразование непосредственно при формировании графика. Это можно сделать с помощью метода `transform_calculate`, который принимает [Vega Expression](https://vega.github.io/vega/docs/expressions/) &ndash; строковое выражение, представляющее собой подмножество операций `JavaScript`:

In [27]:
# Дропнем признак 'Пол'
population = population.drop('Пол', axis=1)

In [28]:
alt.Chart(population).mark_bar().encode(
    x='year:O',
    y='sum(people):Q',
    color='Gender:N'
).transform_calculate(
    Gender='datum.sex == 1 ? "Мужчины" : "Женщины"'
)

alt.Chart(...)

Единственный, возможно, неочевидный момент &ndash; присутствие слова "datum": это стандартный способ обозначить ссылку на текущую строку данных.  
Если же мы предпочитаем формировать выражения средствами `Python`, `Altair` предоставляет API и для этого:

In [ ]:
alt.Chart(population).mark_bar().encode(
    x='year:O',
    y='sum(people):Q',
    color='Пол:N'
).transform_calculate(
    Пол=alt.expr.if_(datum.sex == 1, "Мужчины", "Женщины")
)

> Импортировать метод `if_`:
> ```python
> from altair.expr import if_
> ```
> не получится!

Альтернативный вариант: замена на JavaScript-строку (тернарный оператор):

In [29]:
alt.Chart(population).mark_bar().encode(
    x='year:O',
    y='sum(people):Q',
    color='Пол:N'
).transform_calculate(
    Пол='datum.sex == 1 ? "Мужчины" : "Женщины"'
)

alt.Chart(...)

### Фильтрация

Трансформация фильтрации аналогична по логике. Предположим, что мы хотим построить график, содержащий только мужское население из набора данных переписи. Как и прежде, это можно сделать в `pandas`, но можно выполнять подобные операции и внутри графика.
Для этого используется уже известный нам метод `transform_filter()`:

In [31]:
alt.Chart(population).mark_bar().encode(
    x='year:O',
    y='sum(people):Q',
).transform_filter(
    alt.datum.sex == 1
)

alt.Chart(...)

Ранее мы уже встречали метод `transform_filter`, когда использовали его для фильтрации по результатам выборки (selection).

### Другие виды преобразований

`Altair` поддерживает и множество других методов преобразований. Здесь мы не будем их подробно рассматривать, но примеры доступны в документации: [Vega-Altair &rarr; User Guide &rarr; Data Transformations](https://altair-viz.github.io/user_guide/transform/index.html).

`Altair` предоставляет несколько часто используемых трансформаций, среди которых:

- `transform_aggregate()`
- `transform_bin()`
- `transform_timeunit()`

При этом создаётся новое именованное значение, которое можно многократно использовать в различных частях графика.

Существуют и другие, более специализированные трансформации, например:

- `transform_lookup()`: позволяет выполнять односторонние соединения (join) нескольких наборов данных.
- `transform_window()`: даёт возможность выполнять агрегирование по скользящим окнам, например, вычислять локальные средние значения.

Пример:
> При этом создаётся новое именованное значение, которое можно многократно использовать в различных частях графика.

In [32]:
alt.Chart(cars).mark_point().encode(
    x='Horsepower:Q',
    y='Miles_per_Gallon:Q',
    color='binned_acc:O'  # обращаемся к созданной в `transform_bin` переменной
).transform_bin(
    'binned_acc', 'Acceleration', bin=alt.Bin(maxbins=5)
)

alt.Chart(...)

Для построения многослойных графиков можно определить трансформации в базовом графике и переиспользовать их в других слоях (операторы `+`, `&`, `|`) и кодировках (цвет, текст, оси, подсказки).

## 4.3 Привязки и виджеты

Мы рассмотрели выборки, условия и привязки. Следующий шаг в интерактивной визуализации &ndash; использование графических элементов (*контролов*, *виджетов*) для установления связи между переменной и элементами графика.  
`Altair` предоставляет возможность использования следующих графических элементов ввода:
 - `binding_checkbox` &ndash; флаг, «чекбокс»
 - `binding_radio` &ndash; переключатель, «радиокнопка»
 - `binding_select` &ndash; выпадающий список
 - `binding_range` &ndash; слайдер
 - `binding` &ndash; обобщенный элемент ввода: помимо стандартных элементов ввода (*"checkbox"*, *"radio"*, *"range"*, *"select"*) поддерживает [контролы из HTML](https://developer.mozilla.org/en-US/docs/Web/HTML/Reference/Elements/input) (например, кнопки, выбор цвета из палитры, числовые и текстовые поля ввода и т.п.)

Источники:
 - [Vega-Altair &rarr; API &rarr; binding_checkbox](https://altair-viz.github.io/user_guide/generated/api/altair.binding_checkbox.html)
 - [Vega-Altair &rarr; API &rarr; binding_radio](https://altair-viz.github.io/user_guide/generated/api/altair.binding_radio.html)
 - [Vega-Altair &rarr; API &rarr; binding_select](https://altair-viz.github.io/user_guide/generated/api/altair.binding_select.html)
 - [Vega-Altair &rarr; API &rarr; binding_range](https://altair-viz.github.io/user_guide/generated/api/altair.binding_range.html)
 - [Vega-Altair &rarr; API &rarr; binding](https://altair-viz.github.io/user_guide/generated/api/altair.binding.html)
 - [Vega-Altair &rarr; API &rarr; when](https://altair-viz.github.io/user_guide/generated/api/altair.when.html)
 - [Vega-Altair &rarr; User Guide &rarr; Altair’s Renderer Framework](https://altair-viz.github.io/user_guide/display_frontends.html#altair-s-renderer-framework)
 - [Vega-Altair &rarr; User Guide &rarr; Data-Driven Lookups](https://altair-viz.github.io/user_guide/interactions/bindings_widgets.html?#data-driven-lookups)
 - [Vega-Altair &rarr; User Guide &rarr; Data-Driven Comparisons](https://altair-viz.github.io/user_guide/interactions/bindings_widgets.html?#data-driven-comparisons)
 - [Vega-Altair &rarr; User Guide &rarr; Logic-Driven Comparisons](https://altair-viz.github.io/user_guide/interactions/bindings_widgets.html?#logic-driven-comparisons)
 - [Vega-Altair &rarr; User Guide &rarr; Join Aggregate](https://altair-viz.github.io/user_guide/transform/joinaggregate.html)

 Рассмотрим 3 подхода к использованию виджетов.

 ### Фильтрация по выборке (Data-Driven Lookups)

 Динамическая выборка (поиск) данных &ndash; механизм, при котором одни данные (или действия пользователя) динамически определяют, какие именно строки или значения будут выбраны (отфильтрованы) из другого набора данных.

 #### Выпадающий список

 Используется значение контрола с параметром `selection`:

In [33]:
input_dropdown = alt.binding_select(options=['Europe', 'Japan', 'USA'], name='Region ')
selection = alt.selection_point(fields=['Origin'], bind=input_dropdown)
color = (
    alt.when(selection)
    .then(alt.Color("Origin:N").legend(None))
    .otherwise(alt.value("lightgray"))
)

alt.Chart(cars).mark_point().encode(
    x='Horsepower:Q',
    y='Miles_per_Gallon:Q',
    color=color,
).add_params(
    selection
)

alt.Chart(...)

In [ ]:
cars

В `Altair` условная конструкция `alt.when().then().otherwise()` используется для динамического изменения свойств графиков на основе интерактивного выбора по аналогии с тернарным оператором `if-then-else`. См. [Vega-Altair &rarr; API &rarr; when](https://altair-viz.github.io/user_guide/generated/api/altair.when.html).  
Добавим к значениям выпадающего списка по признаку `Origin` значение «All», которому сопоставим `None`:

In [34]:
input_dropdown = alt.binding_select(options=[None] + sorted(cars.Origin.unique()), labels=['All'] + sorted(cars.Origin.unique()), name='Region ')
selection = alt.selection_point(fields=['Origin'], bind=input_dropdown)
color = (
    alt.when(selection)
    .then(alt.Color("Origin:N"))
    .otherwise(alt.value("lightgray"))
)

alt.Chart(cars).mark_point().encode(
    x='Horsepower:Q',
    y='Miles_per_Gallon:Q',
    color=color,
).add_params(
    selection
)

alt.Chart(...)

> Связать отображение (вкл/выкл) легенды со значением из контрола с использованием `alt.when().then().otherwise()` не получится, поэтому мы убрали вызов `.legend(None)` в `alt.Color("Origin:N").legend(None)`.  

По умолчанию выбран первый элемент из выпадающего списка. Для явной инициализации отображения с определенным значением необходимо задать его в `value=`.

#### Переключатель

Так же как и в предыдущем случае, добавим опцию «All»:

In [35]:
options = ['Europe', 'Japan', 'USA'] + [None]
# options[:-1] - исключим None
labels = [option + ' ' for option in options[:-1]] + ['All']

input_dropdown = alt.binding_radio(
    options=options,
    labels=labels,
    name='Region: '
)
selection = alt.selection_point(
    fields=['Origin'],
    bind=input_dropdown,
)

alt.Chart(cars).mark_point().encode(
    x='Horsepower:Q',
    y='Miles_per_Gallon:Q',
    # Необходимо указать постоянный домен для отображения всей цветовой легенды,
    # даже если выбрано одно значение
    color=alt.Color('Origin:N').scale(domain=options[:-1]),
).add_params(
    selection
).transform_filter(
    selection
)

alt.Chart(...)

#### Текстовое поле ввода

In [37]:
search_input = alt.selection_point(
    fields=['Name'],
    empty=False,  # Start with no points selected
    bind=alt.binding(
        input='search',
        placeholder="Car model",
        name='Search ',
    )
)
alt.Chart(cars).mark_point().encode(
    x='Horsepower:Q',
    y='Miles_per_Gallon:Q',
    tooltip='Name:N',
    opacity=alt.when(search_input).then(alt.value(1)).otherwise(alt.value(0.05)),
).add_params(
    search_input
)

alt.Chart(...)

> Если в поле ввода нажать `Enter`, то поле ввода отправит HTML-форму, и сломается рендер в `VS Code` &ndash; пропадет отображение вывода всех выполненных ячеек.

По умолчанию `Altair` рендерит графики в чистый HTML. Подробнее про режимы рендера можно посмотреть в [Vega-Altair &rarr; User Guide &rarr; Altair’s Renderer Framework](https://altair-viz.github.io/user_guide/display_frontends.html#altair-s-renderer-framework).  
Например, переключение рендера в `alt.renderers.enable("mimetype")` приведет к рендеру графиков в PNG.  
Рендер менять не будем, и жмакать тоже.
> По идее можно отключить событие отправки формы, но доступа к настройкам рендера именно движка IDE под капотом нет. Замена инпута `search` на `text` картину не меняет.

Сначала посмотрим на наши данные:

In [38]:
len(cars.Name.unique())

311

Количество моделей машин достаточно большое, поэтому воспользуемся регулярными выражениями для поиска подстроки (тестовая строка &ndash; `ford|mazda`):

In [39]:
search_input = alt.param(
    value='',
    bind=alt.binding(
        input='search',
        debounce=300,  # Фильтр применится сам через 300 мс после окончания ввода
        placeholder="Car model",
        name='Search ',
    )
)
search_matches = alt.expr.test(alt.expr.regexp(search_input, "i"), alt.datum.Name)

alt.Chart(cars).mark_point().encode(
    x='Horsepower:Q',
    y='Miles_per_Gallon:Q',
    tooltip='Name:N',
    opacity=alt.when(search_matches).then(alt.value(1)).otherwise(alt.value(0.05)),
).add_params(
    search_input
)

alt.Chart(...)

### Сравнения на основе данных (Data-Driven Comparisons)

Фильтрация данных с условием, например, отображение данных по пороговому значению.

#### Слайдер

In [40]:
slider = alt.binding_range(min=0, max=250, step=1, name='HP ')

selector = alt.param(name='SelectorName', value=120, bind=slider)
predicate = alt.datum.Horsepower < selector

alt.Chart(cars).mark_point().encode(
    x='Horsepower:Q',
    y='Miles_per_Gallon:Q',
    tooltip='Name:N',
    color=alt.when(predicate).then(alt.value("blue")).otherwise(alt.value("red")),
).add_params(
    selector
)

alt.Chart(...)

Также в `selection_point` можно добавить поле, связанное со значением слайдера, которое можно использовать повторно:

In [41]:
slider = alt.binding_range(min=0, max=250, step=1, name='HP ')

selector = alt.selection_point(
    name='SelectorName',
    fields=['cutoff'],
    value=[{'cutoff': 120}],
    bind=slider
)

predicate = alt.datum.Horsepower < selector.cutoff

alt.Chart(cars).mark_point().encode(
    x='Horsepower:Q',
    y='Miles_per_Gallon:Q',
    tooltip='Name:N',
    color=alt.when(predicate).then(alt.value("blue")).otherwise(alt.value("red")),
).add_params(
    selector
)

alt.Chart(...)

### Сравнения данных по логическим правилам (Logic-Driven Comparisons)

Графические контролы применяются для изменения отображения графика на основе логических правил. Логические выражения используются для фильтрации данных, условного форматирования или интерактивного выделения элементов.

#### Чекбокс

Флажок будет отвечать за изменение размера маркера в зависимости от размера маркера на шкале легенды `Acceleration`.

In [42]:
bind_checkbox = alt.binding_checkbox(name='Scale point size by "Acceleration": ')
param_checkbox = alt.param(bind=bind_checkbox)

alt.Chart(cars).mark_point().encode(
    x='Horsepower:Q',
    y='Miles_per_Gallon:Q',
    size=alt.when(param_checkbox).then("Acceleration:Q").otherwise(alt.value(25)),
).add_params(
    param_checkbox
)

alt.Chart(...)

#### Color picker

Рассмотрим применение виджета выбора цвета из палитры.

In [43]:
color_usa = alt.param(value="#317bb4", bind=alt.binding(input='color', name='USA '))
color_europe = alt.param(value="#ffb54d", bind=alt.binding(input='color', name='Europe '))
color_japan = alt.param(value="#c12121", bind=alt.binding(input='color', name='Japan '))

alt.Chart(data.cars.url).mark_circle().encode(
    x='Horsepower:Q',
    y='Miles_per_Gallon:Q',
    color=alt.Color(
        'Origin:N',
        scale=alt.Scale(
            domain=['Europe', 'Japan', 'USA'],
            range=[color_europe, color_japan, color_usa]
        )
    )
).add_params(
    color_europe, color_japan, color_usa
)

alt.Chart(...)

#### Условное отображение на основе легенды

In [44]:
selection = alt.selection_point(fields=['Origin'], bind='legend')

alt.Chart(cars).mark_point().encode(
    x='Horsepower:Q',
    y='Miles_per_Gallon:Q',
    color='Origin:N',
    opacity=alt.when(selection).then(alt.value(0.8)).otherwise(alt.value(0.2)),
).add_params(
    selection
)

alt.Chart(...)

#### Панорамирование, масштабирование

In [45]:
selection = alt.selection_interval(bind='scales')

alt.Chart(cars).mark_point().encode(
    x='Horsepower:Q',
    y='Miles_per_Gallon:Q',
    color='Origin:N',
).add_params(
    selection
)

alt.Chart(...)

In [46]:
alt.Chart(cars).mark_point().encode(
    x='Horsepower:Q',
    y='Miles_per_Gallon:Q',
    color='Origin:N',
).interactive()

alt.Chart(...)

### Привязка каналов кодировок к контролам

Пример:
 - создадим выпадающий список с названиями колонок
 - свяжем переменную со значением выпадающего списка
 - посчитаем преобразование для оси х на основе переменной

 Таким образом мы добавим интерактивное отображение признака по оси х.

In [47]:
dropdown = alt.binding_select(
    options=['Horsepower', 'Displacement', 'Weight_in_lbs', 'Acceleration'],
    name='X-axis column '
)
xcol_param = alt.param(
    value='Horsepower',
    bind=dropdown
)

alt.Chart(cars).mark_circle().encode(
    x=alt.X('x:Q').title(''),
    y='Miles_per_Gallon:Q',
    color='Origin:N'
).transform_calculate(
    x=alt.datum[xcol_param]
    # x=f'datum[{xcol_param.name}]'  # Also works
).add_params(
    xcol_param
)

alt.Chart(...)

Добавим слайдер. Движок рендеринга не умеет налету вычислять динамические `min` и `max` для слайдера средствами декларативного синтаксиса (параметры `binding_range` в `Vega-Lite` являются статическими после инициализации). Поэтому слайдер можно рассматривать в качестве фильтра по шкале от $0$ до $1$ или от $0\%$ до $100\%$.  

Хитрость заключается в использовании `Join Aggregate` (см. [Vega-Altair &rarr; User Guide &rarr; Join Aggregate](https://altair-viz.github.io/user_guide/transform/joinaggregate.html)) &ndash; метода преобразования данных `transform_joinaggregate()`. Данное преобразование вычисляет статистику для группировки данных и добавляет вычисленные значения в набор данных (конкатенация). В нашем случае будет вычислено максимальное значение по оси х `max_val="max(x)"` после выбора колонки из выпадающего списка. В качестве предиката будет выступать условие `<значение в строке колонки> меньше <значение в слайдере> * <max_val>`. По значению предиката будем менять размер маркера.

In [48]:
dropdown = alt.binding_select(
    options=['Horsepower', 'Displacement', 'Weight_in_lbs', 'Acceleration'],
    name='X-axis column '
)
xcol_param = alt.param(
    value='Horsepower',
    bind=dropdown
)

slider = alt.binding_range(min=0, max=1, step=0.01, name=f'{xcol_param.param.value} fraction of max ')

selector = alt.selection_point(
    name='SelectorName',
    fields=['cutoff'],
    value=[{'cutoff': 0.5}],
    bind=slider
)

predicate = alt.datum[xcol_param] < selector.cutoff * alt.datum.max_val

alt.Chart(cars).mark_circle().encode(
    x=alt.X('x:Q').title(''),
    y='Miles_per_Gallon:Q',
    color='Origin:N',
    size=alt.when(predicate).then(alt.value(75)).otherwise(alt.value(25)),
).transform_calculate(
    x=alt.datum[xcol_param]
    # x=f'datum[{xcol_param.name}]'  # Also works
).transform_joinaggregate(
    max_val="max(x)"
).add_params(
    selector,
    xcol_param
)

alt.Chart(...)

## 4.4 Настройка графиков

`Altair` предоставляет множество механизмов для настройки внешнего вида графиков. Мы не будем подробно рассматривать все опции, но важно понимать, где и как к ним можно получить доступ.  
В общем случае внешний вид графика можно контролировать в нескольких местах, ниже перечислены уровни в порядке возрастания приоритета.
1. **Настройки верхнего уровня**.  
   На верхнем уровне объекта `Altair` `Chart` мы можем задать настройки, которые будут применяться ко всем панелям и слоям графика.
2. **Локальные настройки**.  
   Настройки верхнего уровня можно локально переопределить, указав соответствующие параметры ближе к месту использования.
3. **Значения в кодировках**.  
   Если значение задано непосредственно в кодировке, оно имеет наивысший приоритет и перекрывает остальные варианты.

Источники:
 - [Vega-Altair &rarr; User Guide &rarr; Top-Level Chart Configuration](https://altair-viz.github.io/user_guide/configuration.html)
 - [Vega-Altair &rarr; User Guide &rarr; Customizing Visualizations](https://altair-viz.github.io/user_guide/customization.html)
 - [Vega-Altair &rarr; User Guide &rarr; Saving Altair Charts](https://altair-viz.github.io/user_guide/saving_charts.html)

Рассмотрим пример.

### Управление свойствами маркеров

Предположим, мы хотим управлять цветом маркеров в диаграмме рассеяния. Рассмотрим все три варианта задания таких настроек.

In [49]:
cars = data.cars.url

alt.Chart(cars).mark_point().encode(
    x='Acceleration:Q',
    y='Horsepower:Q'
)

alt.Chart(...)

#### Настройки верхнего уровня

На верхнем уровне `Altair` предоставляет метод `configure_mark()`, позволяющий настроить множество свойств маркеров в целом, а также свойство `configure_point()`.  
Настройки верхнего уровня стоит воспринимать как «тему» графика: это значения по умолчанию для всех эстетик элементов.  
Настроим некоторые свойства точек через `configure_point`:

In [50]:
alt.Chart(cars).mark_point().encode(
    x='Acceleration:Q',
    y='Horsepower:Q'
).configure_point(
    size=20,
    color='red',
    filled=True
)

alt.Chart(...)

#### Локальные настройки

Внутри метода `mark_point()` мы можем передать локальные параметры, которые *переопределят* верхнеуровневые настройки. Аргументы совпадают с аргументами `configure_mark`.

In [51]:
alt.Chart(cars).mark_point(color='#389561', shape='triangle-down', size=50).encode(
    x='Acceleration:Q',
    y='Horsepower:Q'
).configure_point(
    size=20,
    color='red',
    filled=True
)

alt.Chart(...)

Отметим, что параметры `color`, `size` и `shape` в этом случае переопределяются локальными настройками, а `filled` остаётся таким же, как прежде.

#### Значения в кодировках

Наконец, максимальный приоритет имеют значения, заданные в кодировках. Установим цвет «steelblue» непосредственно в кодировке:

In [52]:
alt.Chart(cars).mark_point(color='#389561', shape='triangle-down', size=50).encode(
    x='Acceleration:Q',
    y='Horsepower:Q',
    color=alt.value('steelblue')
).configure_point(
    size=20,
    color='red',
    filled=True
)

alt.Chart(...)

### Заголовки графика и осей

Заголовки графика и осей по умолчанию формируются на основе источника данных, но иногда возникает необходимость их изменить. Например, рассмотрим гистограмму по тем же данным:

In [53]:
alt.Chart(cars).mark_bar().encode(
    x=alt.X('Acceleration:Q', bin=True),
    y=alt.Y('count()')
)

alt.Chart(...)

Мы можем явно задать подписи осей с помощью аргумента `title` в кодировках:

In [54]:
alt.Chart(cars).mark_bar().encode(
    x=alt.X('Acceleration:Q', bin=True, title='Acceleration (сгруппированные)'),
    y=alt.Y('count()', title='Число элементов в группе')
)

alt.Chart(...)

Аналогично, зададим свойство `title` у самого графика через его свойства:

In [55]:
alt.Chart(cars).mark_bar().encode(
    x=alt.X('Acceleration:Q', bin=True, title='Acceleration (сгруппированные)'),
    y=alt.Y('count()', title='Число элементов в группе')
).properties(
    title='Гистограмма'
)

alt.Chart(...)

### Свойства осей

Если необходимо настроить свойства осей, включая линии сетки, мы можем использовать аргумент `axis` в соответствующих кодировках.

In [56]:
alt.Chart(cars).mark_bar().encode(
    x=alt.X('Acceleration:Q', bin=True, axis=alt.Axis(labelAngle=45)),
    y=alt.Y('count()', axis=alt.Axis(labels=False, ticks=False, title=None))
)

alt.Chart(...)

Заметим, что некоторые из этих значений можно настраивать и на верхнем уровне, если мы хотим применить их ко всему графику. Например:

In [57]:
alt.Chart(cars).mark_bar().encode(
    x=alt.X('Acceleration:Q', bin=True),
    y=alt.Y('count()', axis=alt.Axis(labels=False, ticks=False, title=None))
).configure_axisX(
    labelAngle=45
)

alt.Chart(...)

### Свойства шкал и пределы осей

Каждая кодировка также имеет параметр `scale`, позволяющий управлять пределами осей и другими свойствами шкал.

In [58]:
alt.Chart(cars).mark_point().encode(
    x=alt.X('Acceleration:Q', scale=alt.Scale(domain=[10, 20])),
    y=alt.Y('Horsepower:Q', scale=alt.Scale(domain=[100, 200]))
).configure_point(
    color='#389561',
    shape='triangle-down',
    size=50,
    filled=True
)

alt.Chart(...)

Обратим внимание: если сузить шкалу так, что она станет меньше диапазона данных, точки по умолчанию будут выходить за её пределы. Данные не скрываются по умолчанию в исследовательской визуализации, поскольку это предотвращает случайную потерю точек. Если же мы хотим, чтобы точки обрезались за пределами шкал, зададим свойство `clip` у маркеров.

In [59]:
alt.Chart(cars).mark_point(clip=True).encode(
    x=alt.X('Acceleration:Q', scale=alt.Scale(domain=[10, 20])),
    y=alt.Y('Horsepower:Q', scale=alt.Scale(domain=[100, 200]))
).configure_point(
    color='#389561',
    shape='triangle-down',
    size=50,
    filled=True
).interactive()

alt.Chart(...)

Другой полезный подход &ndash; отображать значения у границ шкалы, чтобы сохранять видимость точек даже при выходе за диапазон:

In [60]:
alt.Chart(cars).mark_point().encode(
    x=alt.X('Acceleration:Q', scale=alt.Scale(domain=[10, 20], clamp=True)),
    y=alt.Y('Horsepower:Q', scale=alt.Scale(domain=[100, 200], clamp=True))
).configure_point(
    color='#389561',
    shape='triangle-down',
    size=50,
    filled=True
).interactive()

alt.Chart(...)

### Цветовые шкалы

Иногда полезно вручную настроить используемую цветовую шкалу.

In [61]:
weather = data.seattle_weather()
weather.head()

,date,precipitation,temp_max,temp_min,wind,weather
0,2012-01-01,0.0,12.8,5.0,4.7,drizzle
1,2012-01-02,10.9,10.6,2.8,4.5,rain
2,2012-01-03,0.8,11.7,7.2,2.3,rain
3,2012-01-04,20.3,12.2,5.6,4.7,rain
4,2012-01-05,1.3,8.9,2.8,6.1,rain


In [62]:
alt.Chart(weather).mark_point().encode(
    x='date:T',
    y='temp_max:Q',
    color='weather:N'
)

alt.Chart(...)

Мы можем изменить цветовую схему, указав свойство шкалы цвета и выбрав любую из [именованных цветовых схем Vega](https://vega.github.io/vega/docs/schemes/#reference):

In [63]:
alt.Chart(weather).mark_point().encode(
    x='date:T',
    y='temp_max:Q',
    color=alt.Color('weather:N', scale=alt.Scale(scheme="dark2"))
)

alt.Chart(...)

Либо создадим собственную цветовую схему, явно задав домен и соответствующий диапазон цветов:

In [64]:
color_scale = alt.Scale(domain=['sun', 'fog', 'drizzle', 'rain', 'snow'],
                        range=['goldenrod', 'gray', 'lightblue', 'steelblue', 'midnightblue'])

alt.Chart(weather).mark_point().encode(
    x='date:T',
    y='temp_max:Q',
    color=alt.Color('weather:N', scale=color_scale) 
)

alt.Chart(...)

### Легенда

Легенда графика добавляется автоматически, если указать значение параметров `color`, `shape` или `size` в `encode()`. Параметров у легенды много ([Vega-Altair &rarr; API &rarr; Legend](https://altair-viz.github.io/user_guide/generated/core/altair.Legend.html)), зададим положение и заголовок:

In [65]:
alt.Chart(cars).mark_point().encode(
    x='Horsepower:Q',
    y='Miles_per_Gallon:Q',
    color=alt.Color('Origin:N').legend(orient="left").title('Регион')
)

alt.Chart(...)

### Темы

`Altair` предоставляет возможность создания кастомных тем оформления и использования встроенных тем (из реестра тем), которые представляют собой глобальные настройки, применяемые к текущей сессии Python.  
Получение текущей темы:

In [66]:
default = alt.theme.get()
default()

{'config': {'view': {'continuousWidth': 300, 'continuousHeight': 300}}}

Тема добавляется в спецификацию графика:

In [67]:
alt.Chart(cars).mark_point().encode(
    x='Horsepower:Q',
    y='Miles_per_Gallon:Q',
    color=alt.Color('Origin:N').legend(orient="left").title('Регион')
).to_dict()

{'config': {'view': {'continuousWidth': 300, 'continuousHeight': 300}},
 'data': {'url': 'https://cdn.jsdelivr.net/npm/vega-datasets@v1.29.0/data/cars.json'},
 'mark': {'type': 'point'},
 'encoding': {'color': {'field': 'Origin',
   'legend': {'orient': 'left'},
   'title': 'Регион',
   'type': 'nominal'},
  'x': {'field': 'Horsepower', 'type': 'quantitative'},
  'y': {'field': 'Miles_per_Gallon', 'type': 'quantitative'}},
 '$schema': 'https://vega.github.io/schema/vega-lite/v6.4.1.json'}

Для изменения темы применяется метод [altair.theme.enable](https://altair-viz.github.io/user_guide/generated/theme/altair.theme.enable.html). Список встроенных тем:

In [68]:
alt.theme.names()

['carbong10',
 'carbong100',
 'carbong90',
 'carbonwhite',
 'dark',
 'default',
 'excel',
 'fivethirtyeight',
 'ggplot2',
 'googlecharts',
 'latimes',
 'none',
 'opaque',
 'powerbi',
 'quartz',
 'urbaninstitute',
 'vox']

In [69]:
alt.theme.enable('urbaninstitute')

alt.Chart(cars).mark_point().encode(
    x='Horsepower:Q',
    y='Miles_per_Gallon:Q',
    color=alt.Color('Origin:N').legend(orient="left").title('Регион')
)

alt.Chart(...)

Применять тему динамически через выпадающий список на самом графике не получится, так как темы задаются статически на уровне рендеринга через `alt.themes.enable()` или через свойства `configure_*()`. Как вариант решения проблемы &ndash; использовать сторонний веб-фреймворк с виджетами, например, библиотеку для создания дашбордов (такую как [Panel](https://github.com/holoviz/panel), [Streamlit](https://streamlit.io/) или [Marimo](https://marimo.io/)).

Для создания кастомной темы используется декоратор [altair.theme.register](https://altair-viz.github.io/user_guide/generated/theme/altair.theme.register.html):

In [70]:
# define, register and enable theme

@alt.theme.register("black_marks", enable=True)
def black_marks() -> alt.theme.ThemeConfig:
    return {
        "config": {
            "view": {"continuousWidth": 300, "continuousHeight": 300},
            "mark": {"color": "black", "fill": "black"},
        }
    }


# draw the chart
cars = data.cars.url
chart = alt.Chart(cars).mark_point().encode(
    x='Horsepower:Q',
    y='Miles_per_Gallon:Q'
)
chart

alt.Chart(...)

В теле функции формируется словарь с конфигурацией параметров оформления графиков. Данный словарь после применения темы записывается в спецификацию графика.

Можно заметить, что тема добавилась в список:

In [71]:
chart.to_dict()

{'config': {'view': {'continuousWidth': 300, 'continuousHeight': 300},
  'mark': {'color': 'black', 'fill': 'black'}},
 'data': {'url': 'https://cdn.jsdelivr.net/npm/vega-datasets@v1.29.0/data/cars.json'},
 'mark': {'type': 'point'},
 'encoding': {'x': {'field': 'Horsepower', 'type': 'quantitative'},
  'y': {'field': 'Miles_per_Gallon', 'type': 'quantitative'}},
 '$schema': 'https://vega.github.io/schema/vega-lite/v6.4.1.json'}

In [72]:
alt.theme.names()

['black_marks',
 'carbong10',
 'carbong100',
 'carbong90',
 'carbonwhite',
 'dark',
 'default',
 'excel',
 'fivethirtyeight',
 'ggplot2',
 'googlecharts',
 'latimes',
 'none',
 'opaque',
 'powerbi',
 'quartz',
 'urbaninstitute',
 'vox']

In [73]:
alt.theme.enable('default')
chart

alt.Chart(...)

## 4.5 Экспорт графиков

У объекта `Chart` есть метод `save()`, который позволяет экспортировать график в различные форматы.  
Подробнее в [Vega-Altair &rarr; User Guide &rarr; Saving Altair Charts](https://altair-viz.github.io/user_guide/saving_charts.html).

### JSON

Фундаментальным представлением графика является формат JSON. В качестве примера возьмем график:

In [74]:
dropdown = alt.binding_select(
    options=['Horsepower', 'Displacement', 'Weight_in_lbs', 'Acceleration'],
    name='X-axis column '
)
xcol_param = alt.param(
    value='Horsepower',
    bind=dropdown
)

chart_for_dump = alt.Chart(cars).mark_circle().encode(
    x=alt.X('x:Q').title(''),
    y='Miles_per_Gallon:Q',
    color='Origin:N'
).transform_calculate(
    x=alt.datum[xcol_param]
    # x=f'datum[{xcol_param.name}]'  # Also works
).add_params(
    xcol_param
)

chart_for_dump

alt.Chart(...)

In [75]:
chart_for_dump.save('data/chart.json')

### HTML

По умолчанию формируется HTML файл без встраивания необходимых JS скриптов (`Vega`, `Vega-Lite`, и `vegaEmbed`).

In [76]:
chart_for_dump.save('data/chart.html')

По умолчанию в `vegaEmbed` используется `canvas` для рендера. Для изменения движка рендера используется параметр `embed_options`:

In [77]:
chart_for_dump.save('data/chart_svg.html', embed_options={'renderer':'svg'})

Можно экспортировать HTML в строковую переменную для дальнейшего использования:

In [ ]:
html_string = chart_for_dump.to_html()
html_string

Для дампа JS скриптов в HTML (офлайн режим) необходимо указать `inline=True` и установить пакет `vl-convert-python`:

In [ ]:
chart_for_dump.save('data/chart_embedded.html', inline=True)

`vl-convert-python` необходим для экспорта в SVG, PNG, PDF форматы.

### SVG



In [78]:
chart_for_dump.save('data/chart.svg')

### PNG

Для экспорта в растровый формат необходимо указать параметр *PPI* (*pixels per inch*), по умолчанию он равен $72$.

In [79]:
chart_for_dump.save('data/chart_300.png', ppi=300)

Для изменения *физического размера* графика используется параметр `scale_factor`:

In [80]:
chart_for_dump.save('data/chart_x2_300.png', ppi=300, scale_factor=2)

### PDF

In [81]:
chart_for_dump.save('data/chart.pdf')